In [26]:
import stim
import numpy as np
import pandas as pd
from sklearn.naive_bayes import GaussianNB
from scipy.stats import multivariate_normal

In [33]:
def regularize_cov(cov, lam=1e-3, shrink=0.1):
    """
    Regularize covariance matrix:
    - lam: diagonal loading (ridge term)
    - shrink: shrinkage toward diagonal
    """
    # Diagonal loading
    cov_reg = cov + lam * np.eye(cov.shape[0])
    
    # Shrinkage toward diagonal
    diag = np.diag(np.diag(cov))
    cov_reg = (1 - shrink) * cov_reg + shrink * diag
    
    return cov_reg

In [47]:
def soft_wrapper(detector_file, logic_file, output_soft_file, output_hard_file, alpha=1.0, log_base='nat', eps=1e-12):
    # Load data
    det_df = pd.read_csv(detector_file, header=None)
    num_detectors = det_df.shape[1]
    det_df.columns = [f"DVal{i+1}" for i in range(num_detectors)]
    logic_df = pd.read_csv(logic_file, header=None)
    logic_df.columns = ["LogicBit"]

    # Compute soft detector features
    # det_df["DetMean"] = det_df.mean(axis=1)
    final_df = pd.DataFrame({"DetMean": det_df.mean(axis=1)})
    final_df["DetVar"] = det_df.var(axis=1)

    # Compute soft logical bit probability
    # Example: logistic function of detector mean
    final_df["LogicProb"] = 1 / (1 + np.exp(-alpha * final_df["DetMean"]))

    # Logic entropy from LogicProb
    p = np.clip(final_df['LogicProb'].values, eps, 1 - eps)
    H_logic = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    if log_base == 'bit':
        H_logic = H_logic / np.log(2)
    final_df['LogicEntropy'] = H_logic

    # Detector entropy from DetVar (rough univariate proxy)
    detvar = np.maximum(final_df['DetVar'].values, eps)
    H_det = 0.5 * np.log(2 * np.pi * np.e * detvar)
    if log_base == 'bit':
        H_det = H_det / np.log(2)
    final_df['DetEntropy'] = H_det

    # Features = detector columns
    X = det_df[[col for col in det_df.columns if col.startswith("DVal")]]
    y = logic_df["LogicBit"]

    # Fit Gaussian model
    model = GaussianNB()
    model.fit(X, y)
    
    # Predict soft probabilities
    probs = model.predict_proba(X)

    # Add to dataframe
    final_df["LogicProb_Naive_Bayes_Gaussian"] = probs[:,1]

    # Estimate parameters for each class
    mu0 = X[y==0].mean(axis=0)
    cov0 = np.cov(X[y==0].T)
    mu1 = X[y==1].mean(axis=0)
    cov1 = np.cov(X[y==1].T)

    # Regularize covariance matrices
    cov0_reg = regularize_cov(cov0, lam=1e-3, shrink=0.1)
    cov1_reg = regularize_cov(cov1, lam=1e-3, shrink=0.1)
    
    # Priors
    p0 = np.mean(y==0)
    p1 = np.mean(y==1)

    # Posterior probabilities
    lik0 = multivariate_normal.pdf(X, mean=mu0, cov=cov0)
    lik1 = multivariate_normal.pdf(X, mean=mu1, cov=cov1)
    post1 = (lik1 * p1) / (lik0 * p0 + lik1 * p1)
    
    final_df["LogicProb_Multivariate_Gaussian"] = post1
    
    # Posterior probabilities
    lik0 = multivariate_normal.pdf(X, mean=mu0, cov=cov0_reg)
    lik1 = multivariate_normal.pdf(X, mean=mu1, cov=cov1_reg)
    post1 = (lik1 * p1) / (lik0 * p0 + lik1 * p1)
    
    final_df["LogicProb_Multivariate_Gaussian_Reg"] = post1

    
    det_df.columns = [f"{i+1}" for i in range(num_detectors)]
    
    # Optionally replace hard bit with probability
    mergedSoft = pd.concat([final_df, logic_df], axis=1)
    mergedHard = pd.concat([det_df, logic_df], axis=1)

    # Save unified dataset
    mergedSoft.to_csv(output_soft_file, index=False)
    mergedHard.to_csv(output_hard_file, index=False)

In [4]:
def gen_dem(distance=3, rounds=5, p=1e-3, shots=20000):
    circ = stim.Circuit.generated("surface_code:rotated_memory_z",
                                  distance=distance, 
                                  rounds=rounds, 
                                  after_clifford_depolarization=p, # Apply depolarizing noise after Clifford gates (idk what those are atm)
                                  before_round_data_depolarization=p, # Apply depolarizing noise before each round
                                  before_measure_flip_probability=p, # Measurement errors
                                  after_reset_flip_probability=p # Reset errors
                                  )
    
    # get detector and observable samples
    sampler = circ.compile_detector_sampler()
    det_samples, obs_samples = sampler.sample(shots=shots, separate_observables=True)
    # det_samples = sampler.sample(shots=shots, separate_observables=False)

    
    # get analog readout wrapper
    measurement_sampler = circ.compile_sampler()
    m2d = circ.compile_m2d_converter()
    measurements = measurement_sampler.sample(shots)
    detectors, observables = m2d.convert(measurements=measurements, separate_observables=True)
    
    return det_samples, obs_samples, measurements, detectors, observables

In [21]:
det_samples, logical_labels, measurement_samples, detectors, observables = gen_dem(distance=3, rounds=5, p=1e-3, shots=20000)

print(f"Detector samples shape: {det_samples.shape}")
print(f"Logical observable sample shape: {logical_labels.shape}")
print(f"Measurement samples shape: {measurement_samples.shape}")
print(f"Detectors shape: {detectors.shape}")
print(f"Observables shape: {observables.shape}")

Detector samples shape: (20000, 40)
Logical observable sample shape: (20000, 1)
Measurement samples shape: (20000, 49)
Detectors shape: (20000, 40)
Observables shape: (20000, 1)


In [12]:
np.savez_compressed('../data/surface_code_data.npz',
                    detectors=det_samples,
                    observables=logical_labels)

In [49]:
np.savetxt('../data/detector_samples.csv', det_samples, delimiter=',', fmt='%d')
np.savetxt('../data/logical_labels.csv', logical_labels, delimiter=',', fmt='%d')
np.savetxt('../data/measurement_samples.csv', measurement_samples, delimiter=',', fmt='%d')
np.savetxt('../data/measurement_detectors.csv', detectors, delimiter=',', fmt='%d')
np.savetxt('../data/observables.csv', observables, delimiter=',', fmt='%d')

In [52]:
# Generated from detector samples and logical labels
soft_wrapper("../data/detector_samples.csv", "../data/logical_labels.csv", "../data/detector_soft_data.csv", "../data/detector_hard_data.csv", alpha=3.0)

# Generated from pure measurement samples and the observables gathered from it
soft_wrapper("../data/measurement_samples.csv", "../data/observables.csv", "../data/measurement_soft_data.csv", "../data/measurement_hard_data.csv", alpha=3.0)

# Generated from detectors gathered from measurement samples and its observables
soft_wrapper("../data/measurement_detectors.csv", "../data/observables.csv", "../data/measurement_detector_soft_data.csv", "../data/measurement_detector_hard_data.csv", alpha=3.0)